In [1]:
%load_ext autoreload
%autoreload 2

#### Uncertaity Quantification
1. Bayesian Uncertainty over a sequence with fixed and dynamic vocabulary
2. Bayesian Uncertainty for 1-hop Markov Chain with fixed and dynamic vocabulary
3. Bayesian Uncertainty for 2-hop Markov Chain with fixed and dynamic vocabulary
4. Bayesian Uncertainty for token-level Markov Chain
5. Combining several Markov Chain based Models

In [20]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.special import beta
from IPython.display import HTML


def explicit_bayesian_update_regularized(sequence, num_points=1000, epsilon=0.01):
    """
    Perform a Bayesian update for binary data with regularization to ensure smooth updates.
    
    Parameters:
    - sequence: list of int (0 or 1), the observed sequence of binary data.
    - num_points: int, number of points to evaluate the posterior over [0, 1].
    - epsilon: float, regularization term to add to prior parameters to ensure smoothness.
    
    Returns:
    - posterior_distributions: list of arrays, posterior probabilities at each step.
    - p_values: array of probabilities over [0, 1].
    """
    # Define a grid of p values
    p_values = np.linspace(0, 1, num_points)
    posterior_distributions = []

    # Start with the uniform prior
    n, m = 0, 0  # Initial counts of 1s and 0s
    alpha = 1 + epsilon
    beta_constant = 1 + epsilon
    for i in range(len(sequence) + 1):  # Include empty sequence
        # Compute the unnormalized posterior: P(p | data) ∝ p^(n+alpha-1) (1-p)^(m+beta-1)
        posterior_unnormalized = p_values**(n + alpha - 1) * (1 - p_values)**(m + beta_constant - 1)
        
        # Normalize the posterior
        normalization_constant = beta(n + alpha, m + beta_constant)  # B(n+alpha, m+beta)
        posterior = posterior_unnormalized / normalization_constant
        posterior_distributions.append(posterior)
        
        # Update counts based on the next observation, if available
        if i < len(sequence):
            if sequence[i] == 1:
                n += 1
            elif sequence[i] == 0:
                m += 1

    return posterior_distributions, p_values

def animate_bayesian_update(sequence, epsilon=0.01):
    """
    Create an animation of Bayesian updates over the cumulative sequence with regularization.
    
    Parameters:
    - sequence: list of int (0 or 1), the observed sequence of binary data.
    - epsilon: float, regularization term to add to prior parameters to ensure smoothness.
    """
    # Get the posterior distributions and p values
    posterior_distributions, p_values = explicit_bayesian_update_regularized(sequence, epsilon=epsilon)

    # Create the figure
    fig, ax = plt.subplots()
    line, = ax.plot([], [], lw=2)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, max(max(posterior) for posterior in posterior_distributions) * 1.2)
    ax.set_title("Bayesian Updates Over Sequence with Regularization")
    ax.set_xlabel("p (P(X=1))")
    ax.set_ylabel("Posterior Density")
    ax.grid()

    def update(frame):
        # Update the line plot for the current step
        line.set_data(p_values, posterior_distributions[frame])
        ax.set_title(f"Step {frame}: Posterior After {frame} Observations")
        return line,

    # Create the animation
    ani = FuncAnimation(fig, update, frames=len(posterior_distributions), blit=True, interval=500)
    plt.close()
    return HTML(ani.to_jshtml())

# Example sequence of 0s and 1s (all 1s case)
sequence = [1,0,1, 1, 1, 1,0,1,1]  
animate_bayesian_update(sequence, epsilon=0.1)

In [26]:
def markov_chain_bayesian_update(sequence, epsilon=0.01):
    """
    Perform Bayesian updates for a 1st-order Markov chain model with proper normalization.
    
    Parameters:
    - sequence: list of int (0 or 1), the observed sequence of binary data.
    - epsilon: float, regularization term to add to prior parameters to ensure smoothness.
    
    Returns:
    - posterior_params: list of dicts, containing normalized posterior probabilities for each step.
    """
    # Initialize counts for transitions with regularization (epsilon)
    transitions = {
        "0->0": [epsilon, epsilon],  # [alpha, beta]
        "0->1": [epsilon, epsilon],
        "1->0": [epsilon, epsilon],
        "1->1": [epsilon, epsilon],
    }
    
    def normalize_transitions(transitions):
        total_0 = transitions["0->0"][0] + transitions["0->1"][0]
        total_1 = transitions["1->0"][0] + transitions["1->1"][0]
        return {
            "P(0|0)": transitions["0->0"][0] / total_0,
            "P(1|0)": transitions["0->1"][0] / total_0,
            "P(0|1)": transitions["1->0"][0] / total_1,
            "P(1|1)": transitions["1->1"][0] / total_1,
        }

    posterior_params = []

    posterior_params.append(normalize_transitions(transitions))

    # Iterate through the sequence and update counts
    for i in range(len(sequence) - 1):
        current = sequence[i]
        next_ = sequence[i + 1]
        if current == 0:
            if next_ == 0:
                transitions["0->0"][0] += 1
            elif next_ == 1:
                transitions["0->1"][0] += 1
        elif current == 1:
            if next_ == 0:
                transitions["1->0"][0] += 1
            elif next_ == 1:
                transitions["1->1"][0] += 1
        posterior_iteration = normalize_transitions(transitions)
        posterior_params.append(posterior_iteration)
    return posterior_params    

def animate_markov_chain(sequence, epsilon=0.01):
    """
    Create an animation of Bayesian updates for a 1st-order Markov chain model with proper normalization.
    
    Parameters:
    - sequence: list of int (0 or 1), the observed sequence of binary data.
    - epsilon: float, regularization term to add to prior parameters to ensure smoothness.
    """
    # Get the posterior parameters over time
    posterior_params = markov_chain_bayesian_update(sequence, epsilon=epsilon)

    # Create the figure
    fig, ax = plt.subplots()
    bars = ax.bar(["P(0|0)", "P(1|0)", "P(0|1)", "P(1|1)"], [0.5, 0.5, 0.5, 0.5])
    ax.set_ylim(0, 1)
    ax.set_title("Markov Chain Learning with Bayesian Updates (Corrected)")
    ax.set_ylabel("Conditional Probability")
    ax.grid(axis="y")

    def update(frame):
        # Update the bar heights for the current step
        params = posterior_params[frame]
        probabilities = [
            params["P(0|0)"],
            params["P(1|0)"],
            params["P(0|1)"],
            params["P(1|1)"],
        ]
        for bar, prob in zip(bars, probabilities):
            bar.set_height(prob)
        ax.set_title(f"Step {frame + 1}: Markov Chain Conditional Probabilities")
        return bars

    # Create the animation
    ani = FuncAnimation(fig, update, frames=len(posterior_params), blit=False, interval=500)
    plt.close()
    return HTML(ani.to_jshtml())

# Example usage:
sequence = [0, 1, 1, 0, 1, 0, 0, 1]  # Example binary sequence
animate_markov_chain(sequence, epsilon=0.01)

In [28]:
def animate_markov_chain_with_pdfs(sequence, epsilon=0.01):
    """
    Create an animation of Bayesian updates for a 1st-order Markov chain model,
    including a second plot for conditional PDFs.
    
    Parameters:
    - sequence: list of int (0 or 1), the observed sequence of binary data.
    - epsilon: float, regularization term to add to prior parameters to ensure smoothness.
    """
    # Get the posterior parameters over time
    posterior_params = markov_chain_bayesian_update(sequence, epsilon=epsilon)

    # Define the grid for the PDFs
    p_grid = np.linspace(0, 1, 100)

    # Create the figure with two subplots
    fig, axs = plt.subplots(2, 1, figsize=(8, 10))
    bar_ax, pdf_ax = axs
    bars = bar_ax.bar(["P(0|0)", "P(1|0)", "P(0|1)", "P(1|1)"], [0.5, 0.5, 0.5, 0.5])
    bar_ax.set_ylim(0, 1)
    bar_ax.set_title("Markov Chain Conditional Probabilities")
    bar_ax.set_ylabel("Probability")
    bar_ax.grid(axis="y")

    pdf_ax.set_xlim(0, 1)
    pdf_ax.set_ylim(0, 10)  # Adjust dynamically later if needed
    pdf_ax.set_title("Conditional PDFs for Transitions")
    pdf_ax.set_xlabel("p (Probability)")
    pdf_ax.set_ylabel("Density")
    pdf_lines = [
        pdf_ax.plot([], [], label="P(0|0)", color="blue")[0],
        pdf_ax.plot([], [], label="P(1|0)", color="orange")[0],
        pdf_ax.plot([], [], label="P(0|1)", color="green")[0],
        pdf_ax.plot([], [], label="P(1|1)", color="red")[0],
    ]
    pdf_ax.legend()

    def update(frame):
        # Update the bar heights for the conditional probabilities
        params = posterior_params[frame]
        probabilities = [
            params["P(0|0)"],
            params["P(1|0)"],
            params["P(0|1)"],
            params["P(1|1)"],
        ]
        for bar, prob in zip(bars, probabilities):
            bar.set_height(prob)
        bar_ax.set_title(f"Step {frame + 1}: Markov Chain Conditional Probabilities")

        # Update the conditional PDFs
        alphas_betas = [
            (params["P(0|0)"] * epsilon, epsilon),
            (params["P(1|0)"] * epsilon, epsilon),
            (params["P(0|1)"] * epsilon, epsilon),
            (params["P(1|1)"] * epsilon, epsilon),
        ]
        for line, (alpha, beta_) in zip(pdf_lines, alphas_betas):
            line.set_data(p_grid, beta.pdf(p_grid, alpha, beta_))

        return bars + pdf_lines

    # Create the animation
    ani = FuncAnimation(fig, update, frames=len(posterior_params), blit=False, interval=500)
    plt.close()
    return HTML(ani.to_jshtml())

# Example usage:
sequence = [0, 1, 1, 0, 1, 0, 0, 1]  # Example binary sequence
animate_markov_chain_with_pdfs(sequence, epsilon=0.01)

AttributeError: 'numpy.ufunc' object has no attribute 'pdf'

In [2]:
import scipy.special as sp

def bayesian_uncertainty(sequence, alpha=1.0, beta=1.0):
    """
    Compute Bayesian uncertainty (posterior entropy) for a binary feature over a sequence.
    
    Parameters:
    - sequence: list of int (0 or 1), the observed sequence of binary data.
    - alpha: float, the prior parameter for 1s in the Beta distribution.
    - beta: float, the prior parameter for 0s in the Beta distribution.
    
    Returns:
    - float, the entropy of the posterior Beta distribution.
    """
    # Count 1s and 0s in the sequence
    n1 = sum(sequence)
    n0 = len(sequence) - n1

    # Update posterior parameters
    alpha_post = alpha + n1
    beta_post = beta + n0

    # Compute the posterior entropy
    a = alpha_post
    b = beta_post
    B_ab = sp.beta(a, b)  # Beta function B(a, b)
    entropy = (
        - (a - 1) * sp.digamma(a)  # (a-1)ψ(a)
        - (b - 1) * sp.digamma(b)  # (b-1)ψ(b)
        + (a + b - 2) * sp.digamma(a + b)  # (a+b-2)ψ(a+b)
        + sp.loggamma(a + b) - sp.loggamma(a) - sp.loggamma(b)  # log(B(a, b))
    )
    return entropy

# Example usage:
sequence = [1, 0, 1, 1, 0, 0, 1]  # Example sequence of 0s and 1s
uncertainty = bayesian_uncertainty(sequence)
print(f"Bayesian uncertainty (posterior entropy): {uncertainty}")

Bayesian uncertainty (posterior entropy): 10.826456269835917
